# Appendix D — Stata, SPSS, R, EViews and Excel → Python Translation Table

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/appD_stata_r_translation.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## D.1 Loading and describing data

## D.2 Selecting and creating variables

## D.3 Collapsing, reshaping, combining

## D.4 Survey data

## D.5 Linear regression

## D.6 Binary outcomes

## D.7 Panel data and causal designs

## D.8 Time series, simulation and optimisation

## D.9 SPSS, EViews and Excel

## D.10 Equivalences you can check

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from indiadata import mospi_cpi, mospi_nas

pc = mospi_nas(25)[["state", "fy", "constant_price"]].rename(
    columns={"constant_price": "pcnsdp"})
cpi = mospi_cpi()

In [ ]:
cpi["year"] = cpi["date"].dt.year
collapsed = cpi.groupby("year", as_index=False)["inflation"].mean()   # collapse (mean)
cpi["year_mean"] = cpi.groupby("year")["inflation"].transform("mean")  # egen mean(), by()
check = cpi.drop_duplicates("year").merge(collapsed, on="year",
                                          suffixes=("_egen", "_collapse"))
print("egen and collapse agree:",
      np.allclose(check["year_mean"].dropna(), check["inflation_collapse"].dropna()))

In [ ]:
wide = pc.pivot(index="state", columns="fy", values="pcnsdp")
long = wide.reset_index().melt(id_vars="state", var_name="fy", value_name="pcnsdp")
back = long.sort_values(["state", "fy"]).reset_index(drop=True)
orig = pc.sort_values(["state", "fy"]).reset_index(drop=True)
print(wide.shape, long.shape, "round trip identical:",
      back.equals(orig[["state", "fy", "pcnsdp"]].astype(back.dtypes.to_dict())))

In [ ]:
d = pc[pc["fy"] == 2023].dropna().copy()
d["growth"] = (pc.set_index(["state", "fy"])["pcnsdp"]
                 .groupby(level="state").pct_change().loc[(d["state"], 2023)]
                 .to_numpy() * 100)
d["ly"] = np.log(d["pcnsdp"])
res = smf.ols("growth ~ ly", data=d).fit(cov_type="HC1")

X = np.column_stack([np.ones(len(d)), d["ly"]])
u = smf.ols("growth ~ ly", data=d).fit().resid.to_numpy()
n, k = X.shape
XtX_inv = np.linalg.inv(X.T @ X)
V = XtX_inv @ (X.T * u**2) @ X @ XtX_inv * n / (n - k)
print(f"{n} states; SE of ly: statsmodels HC1 {res.bse['ly']:.4f}, "
      f"by hand {np.sqrt(V[1, 1]):.4f}")

In [ ]:
from linearmodels.panel import PanelOLS

p = pc.sort_values(["state", "fy"]).copy()
p["g"] = p.groupby("state")["pcnsdp"].pct_change() * 100
p["ly_lag"] = np.log(p.groupby("state")["pcnsdp"].shift(1))
p = p.dropna()
fe = PanelOLS(p.set_index(["state", "fy"])["g"],
              p.set_index(["state", "fy"])[["ly_lag"]], entity_effects=True).fit()
lsdv = smf.ols("g ~ ly_lag + C(state)", data=p).fit()
print(f"FE (PanelOLS): {fe.params['ly_lag']:.4f}   LSDV: {lsdv.params['ly_lag']:.4f}")